# Data Ingestion
For NVIDIA stock market data

In [0]:
import requests
from pyspark.sql import functions as F

API_KEY = 'AEHFJB7KWSYUD2OU'
symbol = 'NVDA'

### Daily stock data

In [0]:
url_stock_data = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={API_KEY}'
r_stock_data = requests.get(url_stock_data)
stock_data = r_stock_data.json()
display(stock_data)

{'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'NVDA',
  '3. Last Refreshed': '2026-02-10',
  '4. Output Size': 'Compact',
  '5. Time Zone': 'US/Eastern'},
 'Time Series (Daily)': {'2026-02-10': {'1. open': '191.3800',
   '2. high': '192.4800',
   '3. low': '188.1200',
   '4. close': '188.5400',
   '5. volume': '136764825'},
  '2026-02-09': {'1. open': '184.2600',
   '2. high': '193.6600',
   '3. low': '183.9500',
   '4. close': '190.0400',
   '5. volume': '196387351'},
  '2026-02-06': {'1. open': '176.6900',
   '2. high': '187.0000',
   '3. low': '174.6000',
   '4. close': '185.4100',
   '5. volume': '231346241'},
  '2026-02-05': {'1. open': '174.9250',
   '2. high': '176.8150',
   '3. low': '171.0300',
   '4. close': '171.8800',
   '5. volume': '206312890'},
  '2026-02-04': {'1. open': '179.4600',
   '2. high': '179.5800',
   '3. low': '171.9100',
   '4. close': '174.1900',
   '5. volume': '207014116'},
  '2026-02-03': {'1. open':

In [0]:
time_series = stock_data["Time Series (Daily)"]

rows = [
    (
        date,
        float(values['1. open']),
        float(values['2. high']),
        float(values['3. low']),
        float(values['4. close']),
        int(values['5. volume'])
    )
    for date, values in time_series.items()
]

columns = ["date", "open", "high", "low", "close", "volume"]
stock_data_df = spark.createDataFrame(rows, columns)

stock_data_df.write.mode("overwrite").saveAsTable("jrvs_dlt.01_bronze.stock_data")
stock_data_df.show()

+----------+-------+--------+--------+------+---------+
|      date|   open|    high|     low| close|   volume|
+----------+-------+--------+--------+------+---------+
|2026-02-10| 191.38|  192.48|  188.12|188.54|136764825|
|2026-02-09| 184.26|  193.66|  183.95|190.04|196387351|
|2026-02-06| 176.69|   187.0|   174.6|185.41|231346241|
|2026-02-05|174.925| 176.815|  171.03|171.88|206312890|
|2026-02-04| 179.46|  179.58|  171.91|174.19|207014116|
|2026-02-03| 186.24|  186.27|  176.23|180.34|202006430|
|2026-02-02|  187.2|   190.3|  184.88|185.61|165794054|
|2026-01-30| 191.21|  194.49|  189.47|191.13|179489463|
|2026-01-29| 191.34|  193.48|  186.06|192.51|171764375|
|2026-01-28| 191.27|  192.35|  189.84|191.52|148552677|
|2026-01-27| 187.24|   190.0|   185.7|188.52|138432307|
|2026-01-26| 187.16|  189.12|  185.99|186.47|124799649|
|2026-01-23|  187.5|   189.6|186.8233|187.67|142748076|
|2026-01-22| 184.75|  186.17|  183.93|184.84|139636626|
|2026-01-21| 179.05| 185.379|   178.4|183.32|200

In [0]:
meta_data = stock_data["Meta Data"]
meta_data_df = spark.createDataFrame([meta_data])
meta_data_df = (
    meta_data_df
    .withColumnsRenamed({
        "1. Information": "information",
        "2. Symbol": "symbol",
        "3. Last Refreshed": "last_refreshed",
        "4. Output Size": "output_size",
        "5. Time Zone": "time_zone"
    })
)
display(meta_data_df)
meta_data_df.write.mode("overwrite").saveAsTable("jrvs_dlt.01_bronze.stock_meta_data")

information,symbol,last_refreshed,output_size,time_zone
"Daily Prices (open, high, low, close) and Volumes",NVDA,2026-02-10,Compact,US/Eastern


### Company info

In [0]:
url_company = f'https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={API_KEY}'
r_company = requests.get(url_company)
company = r_company.json()
display(company)

{'Symbol': 'NVDA',
 'AssetType': 'Common Stock',
 'Name': 'NVIDIA Corporation',
 'Description': 'NVIDIA Corporation is a premier American multinational technology company based in Santa Clara, California, celebrated for its groundbreaking advancements in graphics processing units (GPUs) aimed at gaming and professional markets. As a frontrunner in artificial intelligence and visual computing, NVIDIA plays a vital role in developing transformative technologies, including System on a Chip (SoC) products that enhance mobile computing and revolutionize the automotive sector, particularly in autonomous driving. With a robust and diversified portfolio that encompasses gaming, data centers, and AI infrastructure, NVIDIA stands at the forefront of the evolving tech landscape, consistently driving innovation and performance to meet the growing demands of its global clientele.',
 'CIK': '1045810',
 'Exchange': 'NASDAQ',
 'Currency': 'USD',
 'Country': 'USA',
 'Sector': 'TECHNOLOGY',
 'Industry':

In [0]:
company_df = spark.createDataFrame([company])
display(company_df)
company_df.write.mode("overwrite").saveAsTable("jrvs_dlt.01_bronze.company_data")

200DayMovingAverage,50DayMovingAverage,52WeekHigh,52WeekLow,Address,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongBuy,AnalystRatingStrongSell,AnalystTargetPrice,AssetType,Beta,BookValue,CIK,Country,Currency,Description,DilutedEPSTTM,DividendDate,DividendPerShare,DividendYield,EBITDA,EPS,EVToEBITDA,EVToRevenue,ExDividendDate,Exchange,FiscalYearEnd,ForwardPE,GrossProfitTTM,Industry,LatestQuarter,MarketCapitalization,Name,OfficialSite,OperatingMarginTTM,PEGRatio,PERatio,PercentInsiders,PercentInstitutions,PriceToBookRatio,PriceToSalesRatioTTM,ProfitMargin,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenuePerShareTTM,RevenueTTM,Sector,SharesFloat,SharesOutstanding,Symbol,TrailingPE
170.11,183.81,212.18,86.6,"2788 SAN TOMAS EXPRESSWAY, SANTA CLARA, CA, UNITED STATES, 95051",48,3,1,12,0,253.62,Common Stock,2.314,4.892,1045810,USA,USD,"NVIDIA Corporation is a premier American multinational technology company based in Santa Clara, California, celebrated for its groundbreaking advancements in graphics processing units (GPUs) aimed at gaming and professional markets. As a frontrunner in artificial intelligence and visual computing, NVIDIA plays a vital role in developing transformative technologies, including System on a Chip (SoC) products that enhance mobile computing and revolutionize the automotive sector, particularly in autonomous driving. With a robust and diversified portfolio that encompasses gaming, data centers, and AI infrastructure, NVIDIA stands at the forefront of the evolving tech landscape, consistently driving innovation and performance to meet the growing demands of its global clientele.",4.02,2025-12-26,0.04,0.0002,112696001000,4.02,38.06,24.22,2025-12-04,NASDAQ,January,24.51,131092996000,SEMICONDUCTORS,2025-10-31,4590383137000,NVIDIA Corporation,https://www.nvidia.com,0.632,0.709,46.9,4.330,69.586,38.54,24.53,0.53,0.667,0.625,0.535,1.074,7.67,187141997000,TECHNOLOGY,23330916000,24305000000,NVDA,46.9
